In [1]:
import os
import sys
import yaml
import pandas as pd
import seaborn as sns
current_dir = os.path.dirname(os.path.abspath('.'))
project_root = os.path.abspath(os.path.join(current_dir, "../.."))
sys.path.insert(0, project_root)
import matplotlib.pyplot as plt
plt.style.use('ggplot')

In [12]:
from utils.utils import to_jsonl
from functions.make_dataset import save_data
from functions.model_selection import grid_search_single_model_StratifiedKFold, randomized_single_model_grid_search
from functions.train_model import train_model, save_model, train_voting_model_reg
from functions.evaluate_model import evaluate_reg_model, MetricsOrchestrator
from functions.predict_model import make_prediction_reg
from functions.cross_validate import cross_validate_kfold
from functions.single_model_reg import SingleModelOrchestrator
from functions.voting_model_reg import voting_model,models

In [3]:
with open(os.path.join("../config/config.yaml"), "r") as f:
    config = yaml.safe_load(f)
        
with open(os.path.join( "../config/pipeline.yaml"), "r") as f:
    config_pipe = yaml.safe_load(f)  
    
with open(os.path.join( "../config/pipeline.yaml"), "r") as f:
    config_model = yaml.safe_load(f)

In [4]:
pipeline_name='pipeline1'

In [5]:
# Datasets X_train
X_train = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"X_train_feat_eng_{pipeline_name}.parquet")
)
   
y_train = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"y_train_feat_eng_{pipeline_name}.parquet")
)

# Datasets Y_train
X_val = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"X_val_feat_eng_{pipeline_name}.parquet")
)

y_val = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"y_val_feat_eng_{pipeline_name}.parquet")
)

In [20]:
models_ = [
        "RidgeRegressor",
        "RandomForestRegressor",
        "AdaBoostRegressor",
        "SVRRegressor",
        "GradientBoostingRegressor",
        "XGBRegressor",
        "LGBMRegressor"        
        ]

In [21]:
best_params = voting_model(
    X_train=X_train,
    y_train=y_train,
    model_names=models_,
    search_type="randomized"
    )

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001061 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 820
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 16
[LightGBM] [Info] Start training from score 12.030658
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

In [22]:
model_config=dict(model_name = 'voting')
model_name = 'voting'

In [23]:
model = train_voting_model_reg(
    X_train,
    y_train,
    models=models(model_names=models_),
    best_models_params=best_params,
    model_names=models_,
    search_type="grid",
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_iter=50,
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000142 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 820
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 16
[LightGBM] [Info] Start training from score 12.030658
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

In [24]:
model_info = [{
        'model':model_name,
        'best_paramns': best_params,
        'undersamplig': None,
        'model_type':'voting_model',
        # 'timestamp': datetime.now().isoformat()        
    }]  

In [25]:
model

Pipeline(steps=[('votingregressor',
                 VotingRegressor(estimators=[('RidgeRegressor',
                                              Ridge(alpha=0.3729158771263517,
                                                    random_state=23)),
                                             ('RandomForestRegressor',
                                              RandomForestRegressor(bootstrap=False,
                                                                    criterion='absolute_error',
                                                                    max_depth=9,
                                                                    max_features='sqrt',
                                                                    min_samples_split=6,
                                                                    n_estimators=257,
                                                                    random_state=23)),
                                             ('AdaBoostRegressor',
                                              AdaBoostRegressor(lear...
                                                           multi_strategy=None,
                                                           n_estimators=323,
                                                           n_jobs=-1,
                                                           num_parallel_tree=None, ...)),
                                             ('LGBMRegressor',
                                              LGBMRegressor(colsample_bytree=0.975793880000632,
                                                            learning_rate=0.06102616394446718,
                                                            max_depth=3,
                                                            min_child_samples=5,
                                                            n_estimators=269,
                                                            num_leaves=26,
                                                            random_state=23,
                                                            reg_alpha=0.23937099816163865,
                                                            reg_lambda=0.19868130269462891,
                                                            subsample=0.8439731884605137))]))])

In [26]:
df_cv = cross_validate_kfold(
        X_train, 
        y_train, 
        model,
        model_config,
        score="neg_root_mean_squared_error",
        model_type='voting'
        )
    

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000262 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 818
[LightGBM] [Info] Number of data points in the train set: 934, number of used features: 16
[LightGBM] [Info] Start training from score 12.031714
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

In [27]:
print("\n")
print(df_cv)
print("\n")
print(f"Mean train score {df_cv['scoring'].unique()[0]}: {df_cv['train_score'].mean()} +- {df_cv['train_score'].std()}")
print(f"Mean val score {df_cv['scoring'].unique()[0]}: {df_cv['val_score'].mean()} +- {df_cv['val_score'].std()}")



   fold  train_score  val_score                      scoring model_type  \
0     0      -0.1054    -0.1494  neg_root_mean_squared_error     voting   
1     1      -0.1059    -0.1458  neg_root_mean_squared_error     voting   
2     2      -0.1075    -0.1356  neg_root_mean_squared_error     voting   
3     3      -0.1024    -0.1932  neg_root_mean_squared_error     voting   
4     4      -0.1088    -0.1365  neg_root_mean_squared_error     voting   

    model                   timestamp  
0  voting  2026-05-13T16:34:26.203508  
1  voting  2026-05-13T16:34:26.203508  
2  voting  2026-05-13T16:34:26.203508  
3  voting  2026-05-13T16:34:26.203508  
4  voting  2026-05-13T16:34:26.203508  


Mean train score neg_root_mean_squared_error: -0.10598497360352285 +- 0.0024160309714104343
Mean val score neg_root_mean_squared_error: -0.152120436587938 +- 0.02373101771702997
